# 🐾 Pet Stores Layer – Production Preparation Notebook

## Overview

This notebook finalizes the Pet Stores layer for deployment to the development database.  
It performs the following steps:

1. Load the cleaned Pet Stores dataset  
2. Prepare the final schema for production  
3. Create the `petstores` table in the development database
4. Insert all cleaned records  
5. Validate constraints:
   - Row count  
   - Duplicate ID check  
   - Foreign key validation (`district_id` → `districts`)  
   - Schema consistency  
6. Confirm successful deployment

The goal of this notebook is to ensure that the Pet Stores dataset is fully ready for downstream usage in the Layered Data Pool.



In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import os

In [7]:
engine = create_engine(
    "postgresql+psycopg2://:@localhost:5433/layereddb"
)

print("Connected to production database!")

Connected to production database!


### Load Final Cleaned Dataset (Preserving Leading Zeros)

The final cleaned dataset is loaded from the `petstores_final_export.csv` file.  
Some of the ID fields such as `district_id` and `neighborhood_id` contain leading zeros  
(e.g., `0407`, `0105`).  

When reading CSV files, pandas may interpret these columns as integers, which would  
automatically strip the leading zeros (e.g., `0407` → `407`).  
To avoid this issue, the dataset is explicitly loaded with these ID columns typed as strings.

After loading, the formatting of the IDs is normalized using `zfill()`:

- `neighborhood_id` → always 4 digits  
- `district_id` → always 8 digits  

This ensures consistency before inserting the data into the development database and  
prevents data loss caused by unintended type conversion.


In [3]:
df_final = pd.read_csv(
    "petstores_final_export.csv",
    dtype={"neighborhood_id": str, "district_id": str, "id": str}
)

# Fix ID formats
df_final["neighborhood_id"] = df_final["neighborhood_id"].str.zfill(4)
df_final["district_id"] = df_final["district_id"].str.zfill(8)

df_final.head()

,id,name,brand,opening_hours,phone,website,full_address,longitude,latitude,geometry,district,neighborhood,district_id,neighborhood_id
0,111810614,Fressnapf,Fressnapf,Mo-Fr 09:00-20:00; Sa 09:00-19:00,NaN,NaN,"Fressnapf, 7a, Lützenstraße, Halensee, Charlot...",13.288411,52.499735,POINT (13.288411 52.4997351),Charlottenburg-Wilmersdorf,Halensee,11004004,0407
1,346138343,Fressnapf,Fressnapf,Mo-Fr 09:00-20:00; Sa 09:00-18:00,NaN,NaN,"Fressnapf, 10-11, Müllerstraße, Sprengelkiez, ...",13.367520,52.542519,POINT (13.3675197 52.5425188),Mitte,Wedding,11001001,0105
2,417360251,Fressnapf,Fressnapf,NaN,NaN,NaN,"Fressnapf, 128b, Storkower Straße, Blumenviert...",13.451963,52.533614,POINT (13.4519634 52.5336145),Pankow,Prenzlauer Berg,11003003,0301
3,438385039,Lucas Tierwelt,NaN,Mo-Sa 08:00-20:00,NaN,https://www.lucas-tierwelt.de/,"Lucas Tierwelt, 95, Forckenbeckstraße, Schmarg...",13.308440,52.478876,POINT (13.3084403 52.4788764),Charlottenburg-Wilmersdorf,Schmargendorf,11004004,0403
4,446899194,Das Futterhaus,Das Futterhaus,NaN,NaN,NaN,"Das Futterhaus, 52, Lahnstraße, Richardkiez, N...",13.448016,52.468671,POINT (13.4480157 52.4686707),Neukölln,Neukölln,11008008,0801


### Prepare CREATE TABLE Schema

The schema below defines the final production structure of the Pet Stores table.  
It includes:

- Primary Key constraint on `id`
- Foreign Key constraint on `district_id`
- Structured address fields
- Coordinates (`latitude`, `longitude`)
- Required metadata for downstream BI and geospatial features


In [ ]:
from sqlalchemy import text

create_table_sql = """
DROP TABLE berlin_source_data.petstores CASCADE;


CREATE TABLE berlin_source_data.petstores (
    id VARCHAR(20) PRIMARY KEY,
    name VARCHAR(200) DEFAULT 'Unknown',
    brand VARCHAR(255),
    opening_hours VARCHAR(255),
    phone VARCHAR(255),
    website VARCHAR(500),
    full_address VARCHAR(500),
    longitude DECIMAL(9,6),
    latitude DECIMAL(9,6),
    district VARCHAR(100),
    neighborhood VARCHAR(100),
    district_id VARCHAR(20) NOT NULL,
    neighborhood_id VARCHAR(20),
    geometry VARCHAR,
    CONSTRAINT fk_district
        FOREIGN KEY (district_id)
        REFERENCES berlin_source_data.districts(district_id)
        ON DELETE RESTRICT
        ON UPDATE CASCADE
);
"""

with engine.connect() as conn:
    conn.execute(text(create_table_sql))
    conn.commit()

print("petstores table created successfully!")

### Insert Data Into Production Table

After creating the table, we insert all 82 rows from the cleaned dataset.  
Insertion is performed with `if_exists="append"` to ensure repeatability during development.


In [10]:
df_final.to_sql(
    "petstores",
    engine,
    schema="berlin_source_data",
    if_exists="append",
    index=False,
)

print("Data inserted successfully!")


Data inserted successfully!


### Validation: Row Count

We verify that the number of inserted rows matches the cleaned dataset (expected: 82 rows).


In [11]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM berlin_source_data.petstores"))
    print("Row count in dev table:", result.scalar())


Row count in dev table: 82


### Validation: Duplicate ID Check

Ensures `id` is unique across all rows.  
This is required for primary key consistency and downstream joins.


In [15]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT id, COUNT(*)
        FROM berlin_source_data.petstores
        GROUP BY id
        HAVING COUNT(*) > 1;
    """))
    duplicates = result.fetchall()
    print("Duplicate ID rows:", duplicates)


Duplicate ID rows: []


### Validation: Foreign Key Check

Ensures every `district_id` exists in the `districts` reference table.  
This confirms referential integrity.


In [16]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT p.district_id
        FROM berlin_source_data.petstores p
        LEFT JOIN berlin_source_data.districts d
            ON p.district_id = d.district_id
        WHERE d.district_id IS NULL;
    """))
    missing = result.fetchall()
    print("Invalid district_id rows:", missing)



Invalid district_id rows: []


## Final Deployment Summary

- Pet Stores table successfully created in the **development** database.  
- All 82 rows inserted without errors.  
- No duplicate `id` values detected.  
- All `district_id` values match the reference table.  
- Schema and constraints fully validated.  
- The dataset is now ready for integration into the Layered Data Pool and downstream BI usage.

**Status: Deployment Completed Successfully ✔**

